# Evaluation

In [12]:
from lxml import etree
from rapidfuzz import fuzz
from pathlib import Path
import json


# =====================================================
# 1. PARSING TEI
# =====================================================

def parse_tei(xml_string: str):
    root = etree.fromstring(xml_string.encode())

    entries = []

    for entry in root.xpath(".//entry"):
        lemma = entry.xpath(".//form[@type='lemma']/orth/text()")
        pos = entry.xpath(".//gramGrp/pos/text()")
        definition = entry.xpath(".//sense/def/text()")

        entries.append({
            "lemma": lemma[0].strip() if lemma else "",
            "pos": pos[0].strip() if pos else "",
            "definition": " ".join(definition).strip()
        })

    return entries


# =====================================================
# 2. COMPARAISON FINE
# =====================================================

def analyze_entry(pred, gold):
    lemma_equal = pred["lemma"] == gold["lemma"]
    pos_equal = pred["pos"] == gold["pos"]
    definition_equal = pred["definition"] == gold["definition"]

    definition_similarity = fuzz.ratio(
        pred["definition"], gold["definition"]
    )
    pos_similarity = fuzz.ratio(pred["pos"], gold["pos"])

    errors = {}

    if not lemma_equal:
        errors["lemma"] = {
            "pred": pred["lemma"],
            "gold": gold["lemma"]
        }

    if not pos_equal:
        errors["pos"] = {
            "pred": pred["pos"],
            "gold": gold["pos"],
            "similarity": pos_similarity
        }

    if not definition_equal:
        errors["definition"] = {
            "similarity": definition_similarity,
            "pred_excerpt": pred["definition"][:150],
            "gold_excerpt": gold["definition"][:150]
        }

    return {
        "lemma_equal": lemma_equal,
        "pos_equal": pos_equal,
        "definition_equal": definition_equal,
        "definition_similarity": definition_similarity,
        "pos_similarity": pos_similarity,
        "errors": errors
    }


# =====================================================
# 3. ALIGNEMENT 2 ÉTAPES
# =====================================================

def align_entries(pred, gold, threshold=85):
    matches = []
    replacements = []
    insertions = []
    deletions = []

    used_gold = set()
    unmatched_pred = []

    # -----------------------------
    # PASS 1 — exact match (lemma)
    # -----------------------------
    for p in pred:
        found = False
        for i, g in enumerate(gold):
            if i in used_gold:
                continue

            if p["lemma"] == g["lemma"]:
                matches.append((p, g))
                used_gold.add(i)
                found = True
                break

        if not found:
            unmatched_pred.append(p)

    unmatched_gold = [g for i, g in enumerate(gold) if i not in used_gold]

    # -----------------------------
    # PASS 2 — fuzzy → replacement
    # -----------------------------
    used_gold_fuzzy = set()

    for p in unmatched_pred:
        best_match = None
        best_score = 0
        best_idx = None

        for i, g in enumerate(unmatched_gold):
            if i in used_gold_fuzzy:
                continue

            lemma_score = fuzz.ratio(p["lemma"], g["lemma"])
            def_score = fuzz.ratio(p["definition"], g["definition"])

            score = 0.5 * lemma_score + 0.5 * def_score

            if score > best_score:
                best_score = score
                best_match = g
                best_idx = i

        if best_score >= threshold:
            replacements.append({
                "pred": p,
                "gold": best_match,
                "score": best_score
            })
            used_gold_fuzzy.add(best_idx)
        else:
            insertions.append(p)

    # -----------------------------
    # RESTE = deletions
    # -----------------------------
    deletions = [
        g for i, g in enumerate(unmatched_gold)
        if i not in used_gold_fuzzy
    ]

    return {
        "matches": matches,
        "replacements": replacements,
        "insertions": insertions,
        "deletions": deletions
    }


# =====================================================
# 4. SCORING
# =====================================================

def compute_scores(alignment):
    total = (
        len(alignment["matches"]) +
        len(alignment["replacements"]) +
        len(alignment["insertions"]) +
        len(alignment["deletions"])
    )

    if total == 0:
        return {}

    return {
        "match_rate": len(alignment["matches"]) / total,
        "replacement_rate": len(alignment["replacements"]) / total,
        "insertion_rate": len(alignment["insertions"]) / total,
        "deletion_rate": len(alignment["deletions"]) / total
    }


# =====================================================
# 5. RAPPORT COMPLET
# =====================================================

def build_report(alignment):
    report = {
        "summary": {
            "matches": len(alignment["matches"]),
            "replacements": len(alignment["replacements"]),
            "insertions": len(alignment["insertions"]),
            "deletions": len(alignment["deletions"])
        },
        "matches": [],
        "replacements": [],
        "insertions": [],
        "deletions": []
    }

    # -----------------------------
    # MATCHES (avec erreurs internes)
    # -----------------------------
    for p, g in alignment["matches"]:
        analysis = analyze_entry(p, g)

        report["matches"].append({
            "lemma": g["lemma"],
            "analysis": analysis
        })

    # -----------------------------
    # REPLACEMENTS
    # -----------------------------
    for r in alignment["replacements"]:
        analysis = analyze_entry(r["pred"], r["gold"])

        report["replacements"].append({
            "pred_lemma": r["pred"]["lemma"],
            "gold_lemma": r["gold"]["lemma"],
            "score": r["score"],
            "analysis": analysis
        })

    # -----------------------------
    # INSERTIONS
    # -----------------------------
    for p in alignment["insertions"]:
        report["insertions"].append({
            "lemma": p["lemma"],
            "pos": p["pos"],
            "definition_excerpt": p["definition"][:150]
        })

    # -----------------------------
    # DELETIONS
    # -----------------------------
    for g in alignment["deletions"]:
        report["deletions"].append({
            "lemma": g["lemma"],
            "pos": g["pos"],
            "definition_excerpt": g["definition"][:150]
        })

    return report


# =====================================================
# 6. AFFICHAGE
# =====================================================

def print_report(report, scores):
    print("\n=== SCORES ===")
    for k, v in scores.items():
        print(f"{k}: {v:.2f}")

    print("\n=== SUMMARY ===")
    print(report["summary"])

    print("\n=== MATCHES WITH ERRORS ===")
    for m in report["matches"]:
        if m["analysis"]["errors"]:
            print(f"- {m['lemma']}")
            print("  ERRORS:", m["analysis"]["errors"])

    print("\n=== REPLACEMENTS ===")
    for r in report["replacements"]:
        print(f"- PRED: {r['pred_lemma']}")
        print(f"  GOLD: {r['gold_lemma']}")
        print(f"  ERRORS: {r['analysis']['errors']}")
        print()


# =====================================================
# 7. MAIN
# =====================================================

def evaluation(pred_path, gold_path):
    pred_xml = Path(pred_path).read_text(encoding="utf-8")
    gold_xml = Path(gold_path).read_text(encoding="utf-8")

    pred_entries = parse_tei(pred_xml)
    gold_entries = parse_tei(gold_xml)

    alignment = align_entries(pred_entries, gold_entries)

    scores = compute_scores(alignment)
    report = build_report(alignment)

    print_report(report, scores)

    with open("evaluation_report.json", "w", encoding="utf-8") as f:
        json.dump({
            "scores": scores,
            "report": report
        }, f, indent=2, ensure_ascii=False)


In [ ]:
gt_file = "data/encoded/ground-truth/TR5_p489-490.xml" #TODO: for now it is just a test, just few modifications from the prediction of gpt-5-mini, but it should be the real ground truth
pred_file = "data/encoded/gpt-5-mini/TR5_p489-490.xml"
#pred_file = "data/encoded/gemma-3-27b-it/TR5_p489-490.xml"

evaluation(pred_file, gt_file)


=== SCORES ===
match_rate: 0.33
replacement_rate: 0.04
insertion_rate: 0.62
deletion_rate: 0.00

=== SUMMARY ===
{'matches': 8, 'replacements': 1, 'insertions': 15, 'deletions': 0}

=== MATCHES WITH ERRORS ===
- PRÉSIDENCE
  ERRORS: {'definition': {'similarity': 90.34749034749035, 'pred_excerpt': "La qualité de Président. Praesidis dignitas. La première Présidence d'un tel Parlement est vacante. Il y a force brigue pour cette Présidence.", 'gold_excerpt': "Praesidis dignitas. La première Présidence d'un tel Parlement est vacante. Il y a force brigue pour cette Présidence."}}
- PRÉSIDENT
  ERRORS: {'definition': {'similarity': 18.933012434817485, 'pred_excerpt': "Chef, ou Modérateur d'une Compagnie, d'une Assemblée. Praeses, Moderator. Le Président de l'Assemblée du Clergé, le Président des États. Le plus ancie", 'gold_excerpt': "Chef, ou Modérateur d'une Compagnie, d'une Assemblée. Praeses, Moderator. Le Président de l'Assemblée du Clergé, le Président des États. Le plus ancie"}}
- PR